# All models 尝试包含全部模型

包括:

* Rolling Normal；
* Linear Quantile；
* Fused K1；
* Fused K3；
* Structured K3；

ES 统一采用收益率口径：$ES_\alpha=E[R\mid R\le VaR_\alpha]$，不取负号或绝对值。亏损为负；ES 越低，尾部亏损越严重。




本版采用以下数据流程：

1. 直接读取 `1DataDownload_Clean.ipynb` 生成的两张表，不重复生成 target、momentum 或 rolling features。
2. Rolling Normal 先使用原始 `micro_data` 计算，使用截至预测月（含当月）的最近 192 个日历月。
3. Micro 使用显式特征清单；`target_ret_final` 只作为标签，不进入特征清单。
4. Macro 在初始训练期拟合尺度，之后只使用每个时间块开始前的历史月份更新尺度。
5. 自动识别 local、Colab 和 Kaggle；`quick` 模式仅训练两个 epoch，并运行前两个 OOS 窗口。
6. Linear Quantile、Fused 和 Structured 模型在每个窗口重新初始化，不使用 warm start。

## Planned Robustness Checks

1. Normal versus Student-$t$ components.
2. $K=1$ versus multiple mixture components.
3. Rank versus normal-score micro normalization.
4. With and without the macro/FiLM channel.
5. Rolling versus expanding windows and uniform versus time-decay weights.

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from datetime import datetime
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
env = 'kaggle'

if env == 'kaggle':
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_data = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/micro_data.parquet')
    macro_data = pd.read_parquet('/kaggle/input/datasets/jianbinchenuc/mdndata-1990/macro_data.parquet')
    path = '/kaggle/working'

elif env == 'colab':
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ['KAGGLE_USERNAME'] = 'jianbinchenuc'
    os.environ['KAGGLE_KEY'] = 'fa590c922d3624b8e0734dc7a90dd95a'

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_data = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_data = pd.read_parquet('/content/mdndata/macro_data.parquet') # 可以根据需要切换到 Colab 的路径
    path = '/content/drive/MyDrive/Colab Notebooks/'


elif env == 'local':
    micro_data = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_data = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/macro_data.parquet') # local 路径
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print('Micro DataFrame shape:', micro_data.shape)
print('Micro columns:', micro_data.columns.tolist())
print('Micro months:', micro_data['mthcaldt'].nunique())
display(pd.concat([micro_data.head(5), micro_data.tail(5)]))

print('Macro DataFrame shape:', macro_data.shape)
print('Macro columns:', macro_data.columns.tolist())
print('Macro months:', macro_data['mthcaldt'].nunique())
display(pd.concat([macro_data.head(5), macro_data.tail(5)]))

Micro DataFrame shape: (1305253, 29)
Micro columns: ['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout', 'siccd', 'gvkey', 'public_date', 'bm', 'evm', 'ps', 'pcf', 'opmad', 'cfm', 'roe', 'gprof', 'debt_at', 'short_debt', 'cash_ratio', 'curr_ratio', 'at_turn', 'rd_sale', 'accrual', 'ffi12', 'ffi49', 'linkdt', 'linkenddt']
Micro months: 302


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,ps,pcf,opmad,cfm,roe,gprof,debt_at,short_debt,cash_ratio,curr_ratio,at_turn,rd_sale,accrual,ffi12,ffi49,linkdt,linkenddt
0,10001,1999-12-31,-0.004679,8.50000,2.082500e+04,3.248800e+04,2450,4920,12994,1999-12-31,0.648367,8.382672,0.358747,3.487122,0.056276,0.057297,0.116091,0.105303,0.431281,0.166191,0.028773,1.371188,1.227584,0.000000,-0.092283,8.0,31.0,1986-01-09,2017-08-31
1,10001,2000-01-31,-0.044118,8.12500,1.990625e+04,4.033300e+04,2450,4920,12994,2000-01-31,0.648367,8.382672,0.345460,3.357969,0.056276,0.057297,0.116091,0.105303,0.431281,0.166191,0.028773,1.371188,1.227584,0.000000,-0.092283,8.0,31.0,1986-01-09,2017-08-31
2,10001,2000-02-29,0.015385,8.25000,2.021250e+04,2.217600e+04,2450,4920,12994,2000-02-29,0.627854,8.663359,0.322297,302.541045,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
3,10001,2000-03-31,-0.015288,8.00000,1.971200e+04,7.232000e+04,2464,4920,12994,2000-03-31,0.627854,8.663359,0.313421,294.208955,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
4,10001,2000-04-30,0.011719,8.09375,1.994300e+04,2.634000e+04,2464,4920,12994,2000-04-30,0.627854,8.663359,0.323216,303.402985,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8.0,31.0,1986-01-09,2017-08-31
1305248,93436,2024-09-30,0.221942,261.63000,8.390474e+08,1.604206e+09,3207000,3711,184996,2024-09-30,0.105166,57.713252,8.802612,72.758187,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2.0,23.0,2010-06-29,2099-12-31
1305249,93436,2024-10-31,-0.045025,249.85000,8.020335e+08,1.901431e+09,3210060,3711,184996,2024-10-31,0.105166,57.713252,8.406271,69.482219,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2.0,23.0,2010-06-29,2099-12-31
1305250,93436,2024-11-30,0.381469,345.16000,1.107984e+09,2.082131e+09,3210060,3711,184996,2024-11-30,0.083346,53.762150,11.394010,76.450592,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31
1305251,93436,2024-12-31,0.170008,403.84000,1.298958e+09,1.894644e+09,3216517,3711,184996,2024-12-31,0.083346,53.762150,13.368497,89.698836,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31
1305252,93436,2025-01-31,0.001882,404.60000,1.301403e+09,1.502504e+09,3216517,3711,184996,2025-01-31,0.083346,53.762150,13.393655,89.867643,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2.0,23.0,2010-06-29,2099-12-31


Macro DataFrame shape: (301, 14)
Macro columns: ['mthcaldt', 'PAYEMS', 'UNRATE', 'INDPRO', 'RPI', 'ISRATIOx', 'HOUST', 'BUSLOANS', 'CPIAUCSL', 'VIXCLS', 'TERM_SPREAD', 'CREDIT_SPREAD', 'FEDFUNDS', 'sprtrn']
Macro months: 301


,mthcaldt,PAYEMS,UNRATE,INDPRO,RPI,ISRATIOx,HOUST,BUSLOANS,CPIAUCSL,VIXCLS,TERM_SPREAD,CREDIT_SPREAD,FEDFUNDS,sprtrn
0,1999-12-31,0.001808,0.0,0.002986,0.004062,0.00,7.377759,0.015038,-0.000597,24.64,1.28,1.73,NaN,0.057844
1,2000-01-31,0.002428,0.0,0.004044,0.001568,0.00,7.445418,-0.017497,0.001184,24.95,1.15,1.68,1.88,-0.050904
2,2000-02-29,0.002975,-0.1,0.009730,0.004926,-0.01,7.481556,0.001280,-0.000597,23.37,0.80,1.91,-0.02,-0.020108
3,2000-03-31,0.000330,0.1,0.002812,0.004926,-0.01,7.484930,0.010133,0.003533,24.11,0.31,2.30,0.32,0.09672
4,2000-04-30,0.003188,0.0,0.002750,-0.000210,0.00,7.380256,-0.000968,0.001726,26.20,0.57,2.34,-0.11,-0.030796
296,2024-08-31,0.000718,0.2,-0.006401,0.001581,0.00,7.121252,-0.006629,0.002110,15.00,-1.07,1.69,0.00,0.022835
297,2024-09-30,0.000895,-0.1,0.008110,0.001128,-0.01,7.212294,0.003711,0.000322,16.73,-0.71,1.63,-0.50,0.020197
298,2024-10-31,0.001598,-0.1,-0.002832,0.001122,0.01,7.210818,0.002598,-0.000073,23.16,-0.16,1.46,0.00,-0.009897
299,2024-11-30,0.000075,0.0,-0.002629,0.003545,0.00,7.178545,-0.000157,0.000641,13.51,-0.21,1.44,-0.25,0.057301
300,2024-12-31,0.001426,0.1,-0.001472,0.001561,0.00,7.161622,-0.002535,0.000686,17.35,0.35,1.42,-0.25,-0.02499


In [2]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
# ==========================================
# 1. 全局参数设置
# ==========================================
initial_train_months = 72
val_months = 12
test_months = 12

run_test = 'full'  # 'quick' 或 'full'
industry_type = 'ffi12'  # 'ffi12' 或 'ffi49'

random_seed = 42
seed_everything(random_seed)

micro_normalize = 'rank'  # 'normal' or 'rank' or 'none' for accounting-ratio features
normalization_eps = 1e-8  # Small epsilon to avoid division by zero during normalization

# 每个窗口独立初始化，并使用 Validation early stopping。
early_stopping_max_epochs = 500
batch_size = 1024
lr = 3e-3
patience = 50
dropout_rate = 0.05
sigma_floor = 0.005  # 0.5% monthly component-scale floor

windows_type = 'rolling'      # 'expanding' 或 'rolling'
weights_type = 'month_equal'  # 'uniform', 'month_equal', or 'time_decay'

ALPHAS = (0.05, 0.10)
ALPHA_SUFFIX = {0.05: "5", 0.10: "10"}
ROLLING_SCALE_FLOOR = 1e-6
QUANTILE_LEVELS = np.array([
    0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 
    0.15, 0.25, 0.40, 0.50, 0.60, 0.75, 0.85, 
    0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99
], dtype=float)

Global seed set to 42 to ensure reproducibility.


In [ ]:
# =========================================================
# 核心修复：强制对齐“日历月网格” (Resampling to Continuous Grid)
# =========================================================
print(" -> Reindexing panel data to a continuous monthly grid...")

# 2. 将日期设置为 Index，为 Pandas 的重采样做准备
micro_data = micro_data.set_index('mthcaldt')

# 3. ★ 核心魔法：按 permno 分组，按月 ('ME' 或 'M') 重新采样
# 只要两个日期之间有断层，Pandas 就会自动插入缺失的月份，并将那一行的值设为 NaN
micro_data = micro_data.groupby('permno').resample('ME').asfreq().drop(columns=['permno']).reset_index()
print(" -> Reindexing completed. New shape:", micro_data.shape)
# micro_data = micro_data.drop(columns=['linkdt', 'linkenddt', 'public_date'], errors='ignore') # 删除不再需要的列

 -> Reindexing panel data to a continuous monthly grid...


 -> Reindexing completed. New shape: (1320379, 29)


In [ ]:
# =========================================================
# 数据预处理
# =========================================================
micro_cols = ['mthret', 'mthprc', 'bm', 'evm', 'ps', 'pcf', 'opmad', 'cfm', 'roe', 'gprof', 'debt_at', 
              'short_debt', 'cash_ratio', 'curr_ratio', 'at_turn', 'rd_sale', 'accrual']

micro_data = micro_data.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)
# 1. 计算 t+1 收益率
micro_data['target_ret_final'] = micro_data.groupby('permno')['mthret'].shift(-1)

# 2. 计算过去 12 个月动量（跳过近 1 月，滚动 11 月）
micro_data['log_ret'] = np.log1p(micro_data['mthret'])
micro_data['MOM12m'] = micro_data.groupby('permno')['log_ret'].transform(
    lambda x: x.shift(1).rolling(window=11, min_periods=6).sum()
)
micro_data['MOM12m'] = np.exp(micro_data['MOM12m']) - 1
micro_data = micro_data.drop(columns=['log_ret'])

# 3. 在完整微观时间序列上计算 12 个月滚动均值和标准差
micro_data['mu_bench'] = micro_data.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=6).mean()
)
micro_data['sigma_bench'] = micro_data.groupby('permno')['mthret'].transform(
    lambda x: x.rolling(12, min_periods=6).std()
)

micro_data = micro_data[micro_data[['target_ret_final', 'mthret', 'shrout', 'gvkey']].notna().all(axis=1)]
print(" -> After filtering, new shape:", micro_data.shape)

micro_data['turnover'] = micro_data['mthvol'] / (micro_data['shrout'] * 1000)

# 对mthcap和mthvol取log
micro_data['mthcap_log'] = np.log(micro_data['mthcap'])

derived_micro_cols = ['MOM12m', 'mu_bench', 'sigma_bench', 'turnover', 'mthcap_log']
micro_cols = micro_cols + derived_micro_cols

# 剔除价格低于 5 美元的股票
micro_data = micro_data[micro_data['mthprc'] >= 5].copy()


# 剔除金融行业 (SIC 代码 6000 至 6999) 
micro_data['siccd']=micro_data['siccd'].astype('Int64')
print(f"Total rows before removing financial sector: {len(micro_data)}")
micro_data=micro_data[(micro_data['siccd'] < 6000) | (micro_data['siccd'] > 6999)]
print(f"Total rows after removing financial sector: {len(micro_data)}")
micro_data=micro_data.drop(columns=['siccd'])
print(f"Total duplicates found after merging: {micro_data.duplicated(subset=['permno', 'mthcaldt'], keep=False).sum()}")

# 先用“月份 + 行业”中位数填充，再用当月全市场中位数补充, 最后用0填充缺失值
for col in micro_cols:
    industry_median = micro_data.groupby(['mthcaldt', industry_type])[col].transform('median')
    market_median = micro_data.groupby('mthcaldt')[col].transform('median')
    micro_data[col] = micro_data[col].fillna(industry_median).fillna(market_median).fillna(0)

# 将 industry_type 缺失值填充为 0，并转换为整数类型
micro_data[industry_type] = micro_data[industry_type].fillna(0).astype(int)

micro_data = micro_data[
    micro_data['mthcaldt'].between('2000-01-31', '2024-12-31')
].reset_index(drop=True)

pd.concat([micro_data.head(5), micro_data.tail(5)])


 -> After filtering, new shape: (1248257, 33)
Total rows before removing financial sector: 949545
Total rows after removing financial sector: 725695
Total rows after dropping siccd column: 725695
Total duplicates found after merging: 0


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,gvkey,public_date,bm,evm,ps,pcf,opmad,cfm,roe,gprof,debt_at,short_debt,cash_ratio,curr_ratio,at_turn,rd_sale,accrual,ffi12,ffi49,linkdt,linkenddt,target_ret_final,MOM12m,mu_bench,sigma_bench,turnover,mthcap_log
0,10001,2000-01-31,-0.044118,8.12500,1.990625e+04,4.033300e+04,2450.0,12994,2000-01-31,0.648367,8.382672,0.345460,3.357969,0.056276,0.057297,0.116091,0.105303,0.431281,0.166191,0.028773,1.371188,1.227584,0.000000,-0.092283,8,31,1986-01-09,2017-08-31,0.015385,0.000000,0.000000,0.000000,16.462449,9.898789
1,10001,2000-02-29,0.015385,8.25000,2.021250e+04,2.217600e+04,2450.0,12994,2000-02-29,0.627854,8.663359,0.322297,302.541045,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8,31,1986-01-09,2017-08-31,-0.015288,0.000000,0.000000,0.000000,9.051429,9.914057
2,10001,2000-03-31,-0.015288,8.00000,1.971200e+04,7.232000e+04,2464.0,12994,2000-03-31,0.627854,8.663359,0.313421,294.208955,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8,31,1986-01-09,2017-08-31,0.011719,0.000000,0.000000,0.000000,29.350649,9.888983
3,10001,2000-04-30,0.011719,8.09375,1.994300e+04,2.634000e+04,2464.0,12994,2000-04-30,0.627854,8.663359,0.323216,303.402985,0.049894,0.048829,0.096569,0.099794,0.440956,0.213833,0.024975,1.282040,1.292765,0.000000,0.026443,8,31,1986-01-09,2017-08-31,-0.023166,0.000000,0.000000,0.000000,10.689935,9.900633
4,10001,2000-05-31,-0.023166,7.90625,1.948100e+04,2.211600e+04,2464.0,12994,2000-05-31,0.721287,8.026882,0.285656,7.736264,0.051604,0.046793,0.106189,0.106409,0.446801,0.242687,0.022775,1.209732,1.389639,0.000000,-0.021281,8,31,1986-01-09,2017-08-31,0.027607,0.000000,-0.010025,0.022404,8.975649,9.877195
721923,93436,2024-08-31,-0.077390,214.11000,6.840044e+08,1.610837e+09,3194640.0,184996,2024-08-31,0.105166,57.713252,7.174588,59.301712,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2,23,2010-06-29,2099-12-31,0.221942,-0.100783,-0.006184,0.139169,504.231005,20.343475
721924,93436,2024-09-30,0.221942,261.63000,8.390474e+08,1.604206e+09,3207000.0,184996,2024-09-30,0.105166,57.713252,8.802612,72.758187,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2,23,2010-06-29,2099-12-31,-0.045025,-0.144312,0.014849,0.153502,500.220287,20.547778
721925,93436,2024-10-31,-0.045025,249.85000,8.020335e+08,1.901431e+09,3210060.0,184996,2024-10-31,0.105166,57.713252,8.406271,69.482219,0.075809,0.183071,0.217032,0.207100,0.095088,0.293684,1.017029,1.760274,0.902169,0.047001,0.010123,2,23,2010-06-29,2099-12-31,0.381469,0.302679,0.027542,0.140071,592.335083,20.502661
721926,93436,2024-11-30,0.381469,345.16000,1.107984e+09,2.082131e+09,3210060.0,184996,2024-11-30,0.083346,53.762150,11.394010,76.450592,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2,23,2010-06-29,2099-12-31,0.170008,0.040696,0.043050,0.167882,648.626898,20.825808
721927,93436,2024-12-31,0.170008,403.84000,1.298958e+09,1.894644e+09,3216517.0,184996,2024-12-31,0.083346,53.762150,13.368497,89.698836,0.084756,0.183922,0.208738,0.203448,0.099842,0.274887,1.047903,1.798235,0.866390,0.044858,-0.014637,2,23,2010-06-29,2099-12-31,0.001882,0.389088,0.054301,0.171772,589.035974,20.984828


In [ ]:
### ==========================================
### Rolling Normal：使用截至预测月的最近 max_history_months 个日历月
### ==========================================

def normal_var_and_es(alpha, pred_mean, pred_vol):
    """计算 Normal distribution 的左尾 VaR 和 收益率 ES。"""
    z_alpha = norm.ppf(alpha)
    var_value = pred_mean + pred_vol * z_alpha
    es_return = pred_mean - pred_vol * norm.pdf(z_alpha) / alpha
    return float(var_value), float(es_return)


def run_rolling_normal(panel, max_history_months):
    """
    对每只股票使用截至预测月的最近 max_history_months 个日历月估计 Normal distribution。
    历史窗口包含预测月；股票缺月时不会用更早的观测补足 max_history_months 条。
    """
    panel = panel.sort_values(["permno", "mthcaldt"]).reset_index(drop=True).copy()
    panel["mthcaldt"] = pd.to_datetime(panel["mthcaldt"]) + pd.offsets.MonthEnd(0)

    # 将日期转为连续月份编号，便于严格按日历月确定每只股票的左边界。
    month_id = panel["mthcaldt"].dt.year.to_numpy() * 12 + panel["mthcaldt"].dt.month.to_numpy()
    returns = panel["mthret"].to_numpy(dtype=float)
    pred_mean = np.full(len(panel), np.nan)
    pred_vol = np.full(len(panel), np.nan)
    history_n = np.zeros(len(panel), dtype=np.int32)

    for positions in panel.groupby("permno", sort=False).indices.values():
        positions = np.asarray(positions)
        stock_months = month_id[positions]
        stock_returns = returns[positions]

        # 对月份 t，使用 [t-max_history_months+1, t] 的收益率，共 max_history_months 个日历月。
        left = np.searchsorted(stock_months, stock_months - max_history_months + 1, side="left")
        right = np.arange(len(positions)) + 1  # 包含当月
        count = right - left

        cumulative_sum = np.r_[0.0, np.cumsum(stock_returns)]
        cumulative_sq_sum = np.r_[0.0, np.cumsum(stock_returns ** 2)]
        window_sum = cumulative_sum[right] - cumulative_sum[left]
        window_sq_sum = cumulative_sq_sum[right] - cumulative_sq_sum[left]

        has_mean = count > 0
        has_vol = count > 1
        pred_mean[positions[has_mean]] = window_sum[has_mean] / count[has_mean]
        variance = (window_sq_sum[has_vol] - window_sum[has_vol] ** 2 / count[has_vol]) / (count[has_vol] - 1)
        pred_vol[positions[has_vol]] = np.sqrt(np.maximum(variance, 0.0))
        history_n[positions] = count

    panel["pred_mean"] = pred_mean
    panel["pred_vol"] = pred_vol
    panel["history_n"] = history_n

    # 新上市股票只有当月收益时，均值使用该收益，波动率使用当月横截面标准差。
    monthly_fallback = panel.groupby("mthcaldt")["mthret"].agg(["mean", "std"])
    panel["pred_mean"] = panel["pred_mean"].fillna(panel["mthcaldt"].map(monthly_fallback["mean"]))
    panel["pred_vol"] = panel["pred_vol"].fillna(panel["mthcaldt"].map(monthly_fallback["std"]))
    panel["pred_vol"] = panel["pred_vol"].clip(lower=ROLLING_SCALE_FLOOR)

    available_dates = pd.DatetimeIndex(np.sort(panel["mthcaldt"].unique()))
    oos_panel = panel[panel["mthcaldt"] >= available_dates[max_history_months]].copy()
    if run_test == "quick":
        quick_dates = np.sort(oos_panel["mthcaldt"].unique())[:2 * test_months]
        oos_panel = oos_panel[oos_panel["mthcaldt"].isin(quick_dates)].copy()

    output_rows = []
    for _, row in tqdm(oos_panel.iterrows(), total=len(oos_panel), desc="Rolling Normal"):
        output_row = {
            "permno": row["permno"], "mthcaldt": row["mthcaldt"],
            "target_ret_final": float(row["target_ret_final"]), "model_id": "rolling_normal",
            "pred_mean": float(row["pred_mean"]), "pred_vol": float(row["pred_vol"]),
            "history_n": int(row["history_n"]),
        }
        for alpha in ALPHAS:
            suffix = ALPHA_SUFFIX[alpha]
            var_value, es_return = normal_var_and_es(alpha, output_row["pred_mean"], output_row["pred_vol"])
            output_row[f"var_{suffix}"] = var_value
            output_row[f"es_{suffix}"] = es_return
        output_rows.append(output_row)

    return pd.DataFrame(output_rows).sort_values(["mthcaldt", "permno"]).reset_index(drop=True)


rolling_normal_forecasts = run_rolling_normal(
    panel=micro_data, max_history_months=initial_train_months + val_months
)

# rolling_normal_forecasts.attrs["es_convention"] = "return"
rolling_normal_forecasts.to_parquet(
    os.path.join(path, f'rolling_normal_forecasts_{timestamp}.parquet'), index=False
)

display(pd.concat([rolling_normal_forecasts.head(4), rolling_normal_forecasts.tail(4)]))

rolling_normal_summary = rolling_normal_forecasts.groupby("mthcaldt").agg(
    n_firms=("permno", "size"), mean_history_n=("history_n", "mean"),
    mean_predicted_vol=("pred_vol", "mean"), mean_es_5=("es_5", "mean"),
)
display(rolling_normal_summary.head())

In [ ]:
# =========================================================
# 1. Validate the cleaned micro table and normalize explicit features
# =========================================================
micro_data = micro_data[['permno', 'mthcaldt', industry_type, 'target_ret_final'] + micro_cols].copy()
micro_data['mthcaldt'] = pd.to_datetime(micro_data['mthcaldt']) + pd.offsets.MonthEnd(0)

# 每月截面 1%/99% winsorize
for col in micro_cols:
    grouped = micro_data.groupby("mthcaldt")[col]
    lower = grouped.transform("quantile", q=0.01)
    upper = grouped.transform("quantile", q=0.99)
    micro_data[col] = micro_data[col].clip(lower=lower, upper=upper)

if micro_normalize == 'rank':
    micro_norm = (
        micro_data.groupby('mthcaldt')[micro_cols].rank(pct=True) * 2.0 - 1.0
    )
elif micro_normalize == 'normal':
    monthly_mean = micro_data.groupby('mthcaldt')[micro_cols].transform('mean')
    monthly_std = micro_data.groupby('mthcaldt')[micro_cols].transform('std')
    micro_norm = (
        micro_data[micro_cols] - monthly_mean
    ) / (monthly_std + normalization_eps)
elif micro_normalize == 'none':
    micro_norm = micro_data[micro_cols]
else:
    raise ValueError("micro_normalize must be 'normal', 'rank', or 'none'")

micro_data[micro_cols] = micro_norm.fillna(0.0).to_numpy()

micro_data = micro_data.sort_values(['mthcaldt', 'permno']).reset_index(drop=True)

print('Micro model panel:', micro_data.shape)

display(pd.concat([micro_data.head(3), micro_data.tail(3)]))

In [ ]:
def standardize_macro_window(
    train_df, val_df, test_df, macro_cols,
    date_col="mthcaldt",
):
    # 每个月的宏观数据只保留一行，避免按公司数量加权
    train_macro = (
        train_df[[date_col] + macro_cols]
        .drop_duplicates()
        .sort_values(date_col)
    )

    # 同一个月不应该出现两组不同的宏观数据
    if train_macro[date_col].duplicated().any():
        raise ValueError("同一个月份存在不一致的宏观数据。")

    train_median = train_macro[macro_cols].median()

    if train_median.isna().any():
        bad_cols = train_median[train_median.isna()].index.tolist()
        raise ValueError(f"Training 中完全缺失的宏观变量: {bad_cols}")

    train_x = train_macro[macro_cols].fillna(train_median)
    train_mean = train_x.mean()

    train_std = train_x.std(ddof=0)
    train_std = train_std.mask(train_std < 1e-8, 1.0)

    def transform(df):
        return (
            df[macro_cols].fillna(train_median) - train_mean
        ) / train_std

    scaler = {
        "median": train_median,
        "mean": train_mean,
        "std": train_std,
    }

    return (
        transform(train_df),
        transform(val_df),
        transform(test_df),
        scaler,
    )

In [ ]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
macro_cols = [col for col in macro_data.columns if col != 'mthcaldt']
macro_data['mthcaldt'] = pd.to_datetime(macro_data['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_data = macro_data.sort_values('mthcaldt').reset_index(drop=True)

processed_df = (
    micro_data
    .merge(macro_data, on='mthcaldt', how='left', validate='many_to_one')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

pd.concat([processed_df.head(5), processed_df.tail(5)])

### ==========================================
### Linear Quantile：线性分位数回归
### ==========================================

In [ ]:
linear_batch_size = 8192
linear_lr = 1e-3
linear_weight_decay = 1e-4

In [ ]:
def tensor_batch_indices(length, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(length, device=device)
    else:
        indices = torch.arange(length, device=device)
    for start in range(0, length, batch_size):
        yield indices[start:start + batch_size] 

In [ ]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col

    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months  < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months < len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

def make_window_generator(panel):
    generator = DataWindowGenerator(df=panel, date_col="mthcaldt")

    if windows_type == "expanding":
        return generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    return generator.rolling_window_generator(
        train_months=initial_train_months,
        val_months=val_months,
        test_months=test_months,
    )

In [ ]:
# ==========================================
# Linear Quantile 模型
# Macro + Micro + FF49 行业固定效应
# ==========================================
class LinearQuantile(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, quantile_levels):
        super().__init__()
        self.num_industries = num_industries
        input_dim = macro_dim + micro_dim + num_industries - 1
        self.linear = nn.Linear(input_dim, len(quantile_levels))

        quantiles = torch.tensor(quantile_levels, dtype=torch.float32).view(1, -1)
        self.register_buffer("quantile_levels", quantiles)

    def forward(self, macro, micro, industry):
        industry_dummy = F.one_hot(
            industry.long(), num_classes=self.num_industries
        ).to(macro.dtype)[:, 1:]

        x = torch.cat([macro, micro, industry_dummy], dim=1)
        return self.linear(x)

In [ ]:
def weighted_pinball_loss(pred, target, weight, quantile_levels):
    target = target.reshape(-1, 1)
    weight = weight.reshape(-1)

    error = target - pred
    loss = torch.maximum(
        quantile_levels * error,
        (quantile_levels - 1.0) * error,
    )
    loss_per_stock = loss.mean(dim=1)

    return (
        (loss_per_stock * weight).sum()
        / weight.sum().clamp_min(1e-12)
    )

In [ ]:
# ==========================================
# Sample weights: prevent months with more listed stocks from dominating
# ==========================================
def calculate_sample_weights(df_dates, mode='month_equal', half_life_months=36):
    dates = pd.to_datetime(df_dates).reset_index(drop=True)
    if mode == 'uniform':
        weights = np.ones(len(dates), dtype=np.float64)
    elif mode in {'month_equal', 'time_decay'}:
        month_counts = dates.groupby(dates).transform('size').to_numpy(dtype=float)
        weights = 1.0 / np.maximum(month_counts, 1.0)
        if mode == 'time_decay':
            max_date = dates.max()
            months_diff = (
                (max_date.year - dates.dt.year) * 12
                + (max_date.month - dates.dt.month)
            ).to_numpy(dtype=float)
            weights *= np.power(0.5, months_diff / half_life_months)
    else:
        raise ValueError("weights_type must be 'uniform', 'month_equal', or 'time_decay'")
    weights = weights / weights.mean()
    return weights.astype(np.float32)

# ==========================================
# 将完整面板转成设备端 Tensor
# ==========================================
def make_device_tensors(panel):
    X_micro = torch.from_numpy(
        panel[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    industry = torch.from_numpy(
        panel[industry_type].to_numpy(dtype=np.int64, copy=True)
    ).to(device)

    target = torch.from_numpy(
        panel["target_ret_final"].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    return X_micro, industry, target

In [ ]:
# ==========================================
# Linear Quantile 训练
# ==========================================
def train_linear_quantile(
    model,
    X_train_macro, X_train_micro, X_train_industry, y_train, w_train,
    X_val_macro, X_val_micro, X_val_industry, y_val, w_val,
    epochs=300, batch_size=8192, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=linear_weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
        mode="min", factor=0.5, patience=5, threshold=1e-4, threshold_mode="abs", min_lr=1e-6,
    )

    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    train_size = len(y_train)

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            train_size,
            batch_size,
            X_train_macro.device,
            shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()
            pred = model(b_macro, b_micro, b_industry)
            loss = weighted_pinball_loss(
                pred,
                b_y,
                b_w,
                model.quantile_levels,
            )
            loss.backward()
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_pred = model(
                X_val_macro,
                X_val_micro,
                X_val_industry,
            )
            val_loss = weighted_pinball_loss(
                val_pred,
                y_val,
                w_val,
                model.quantile_levels,
            ).item()

        if not np.isfinite(val_loss):
            raise RuntimeError("Linear Quantile validation loss became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return (
        model,
        best_val_loss,
        train_loss_history,
        val_loss_history,
    )

In [ ]:
# ==========================================
# 从预测分位数计算均值、波动率、VaR 和 ES
# ==========================================
def add_quantile_distribution_results(output, quantile_pred):
    levels = QUANTILE_LEVELS
    quantile_pred = np.sort(quantile_pred, axis=1)
    trapz = np.trapezoid

    full_levels = np.r_[0.0, levels, 1.0]
    full_quantiles = np.column_stack([
        quantile_pred[:, 0],
        quantile_pred,
        quantile_pred[:, -1],
    ])

    pred_mean = trapz(full_quantiles, full_levels, axis=1)
    second_moment = trapz(full_quantiles ** 2, full_levels, axis=1)

    output["pred_mean"] = pred_mean
    output["pred_vol"] = np.sqrt(
        np.maximum(second_moment - pred_mean ** 2, 0.0)
    )

    for alpha in ALPHAS:
        suffix = ALPHA_SUFFIX[alpha]
        alpha_idx = np.flatnonzero(np.isclose(levels, alpha))[0]
        below_alpha = levels < alpha

        var_value = quantile_pred[:, alpha_idx]
        tail_levels = np.r_[0.0, levels[below_alpha], alpha]
        tail_quantiles = np.column_stack([
            quantile_pred[:, 0],
            quantile_pred[:, below_alpha],
            var_value,
        ])

        output[f"var_{suffix}"] = var_value
        output[f"es_{suffix}"] = (
            trapz(tail_quantiles, tail_levels, axis=1) / alpha
        )

    output["quantile_levels"] = json.dumps(levels.tolist())
    output["quantile_vec"] = [
        json.dumps(row.tolist()) for row in quantile_pred
    ]

    return output

In [ ]:
# ==========================================
# Linear Quantile 滚动样本外预测
# ==========================================
def run_linear_quantile(panel):
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)
    window_gen = make_window_generator(panel)

    global_X_mic, global_ind, global_Y = make_device_tensors(panel)
    
    all_predictions = []
    history_logs = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()
        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        train_macro_z, val_macro_z, test_macro_z, macro_scaler = (
            standardize_macro_window(
                train_df,
                val_df,
                test_df,
                macro_cols,
            )
        )

        X_tr_mac = torch.from_numpy(
            train_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_va_mac = torch.from_numpy(
            val_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_te_mac = torch.from_numpy(
            test_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_tr_mic, X_tr_ind, Y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        X_va_mic, X_va_ind, Y_va = global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]


        w_tr = torch.from_numpy(
            calculate_sample_weights(
                train_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(
                val_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        # 每个窗口重新初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = LinearQuantile(
            macro_dim=len(macro_cols),
            micro_dim=len(micro_cols),
            num_industries=50,
            quantile_levels=QUANTILE_LEVELS,
        ).to(device)

        # 使用训练收益率的无条件分位数初始化截距
        initial_quantiles = np.quantile(
            train_df["target_ret_final"].to_numpy(),
            QUANTILE_LEVELS,
        )

        with torch.no_grad():
            model.linear.weight.zero_()
            model.linear.bias.copy_(
                torch.tensor(
                    initial_quantiles,
                    dtype=torch.float32,
                    device=device,
                )
            )

        model, best_val_loss, tr_hist, va_hist = train_linear_quantile(
            model=model,
            X_train_macro=X_tr_mac,
            X_train_micro=X_tr_mic,
            X_train_industry=X_tr_ind,
            y_train=Y_tr,
            w_train=w_tr,
            X_val_macro=X_va_mac,
            X_val_micro=X_va_mic,
            X_val_industry=X_va_ind,
            y_val=Y_va,
            w_val=w_va,
            epochs=early_stopping_max_epochs,
            batch_size=linear_batch_size,
            lr=linear_lr,
            patience=patience,
        )

        model.eval()
        with torch.no_grad():
            quantile_pred = model(
                X_te_mac,
                X_te_mic,
                X_te_ind,
            ).cpu().numpy()

        output = test_df[
            ["permno", "mthcaldt", "target_ret_final"]
        ].copy()

        output["model_id"] = "linear_quantile"
        output["validation_loss"] = best_val_loss
        output["epochs_run"] = len(tr_hist)

        output = add_quantile_distribution_results(
            output,
            quantile_pred,
        )
        all_predictions.append(output)

        history_logs.append({
            "window": window_idx,
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "best_epoch": int(np.argmin(va_hist) + 1),
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)

    return forecasts, metrics

In [ ]:
linear_quantile_forecasts, linear_quantile_metrics = run_linear_quantile(processed_df)
# 执行保存
linear_quantile_forecasts.to_parquet(os.path.join(path, f'linear_quantile_forecasts_{timestamp}.parquet'), index=False)
linear_quantile_metrics.to_parquet(os.path.join(path, f'linear_quantile_metrics_{timestamp}.parquet'), index=False)

display(pd.concat([linear_quantile_forecasts.head(4),linear_quantile_forecasts.tail(4),]))
display(pd.concat([linear_quantile_metrics.head(4),linear_quantile_metrics.tail(4),]))

### ==========================================
### Fused models (K=1 and 3)
### ==========================================

In [ ]:
# ==========================================
# Fused MDN
# K=1：Conditional Normal
# K=3：Fused Normal Mixture
# ==========================================
class FusedMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6,
        hidden_dim=64, head_dim=8, num_components=1, dropout_rate=0.05, dist_type="normal",
    ):
        super().__init__()

        self.num_components = num_components
        self.dist_type = dist_type
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        input_dim = macro_dim + micro_dim + ind_emb_dim

        self.fused_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(head_dim, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        # 行业 embedding 不要初始化得太大
        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)

        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)

        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            # K=3 初始概率相同
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        industry_embedding = self.ind_embedding(x_industry)

        x = torch.cat([x_macro, x_micro, industry_embedding], dim=1,)
        h = self.fused_extractor(x)

        if self.pi_head is None:
            pi_logits = h.new_zeros((len(h), 1))
        else:
            pi_logits = self.pi_head(h)

        mu = self.mu_head(h)
        sigma = F.softplus(self.sigma_head(h)) + sigma_floor

        if self.nu_head is None:
            nu = None
        else:
            nu = F.softplus(self.nu_head(h)) + 2.01

        return pi_logits, mu, sigma, nu

In [ ]:
# ==========================================
# 所有 MDN 共用的加权 NLL
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type="normal",):
    target = target.reshape(-1, 1)
    weights = weights.reshape(-1)

    log_pi = torch.log_softmax(pi_logits, dim=1)

    if dist_type == "normal":
        dist = torch.distributions.Normal(mu, sigma)
    elif dist_type == "t":
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma,)
    else:
        raise ValueError("dist_type must be 'normal' or 't'")

    log_component = dist.log_prob(target.expand_as(mu))
    observation_nll = -torch.logsumexp( log_pi + log_component, dim=1,)

    return (observation_nll * weights).sum() / weights.sum().clamp_min(1e-12)

In [ ]:
# ==========================================
# 所有 MDN 共用的 Validation
# ==========================================
def evaluate_mdn(model, X_macro, X_micro, X_industry, target, weights, batch_size,):
    model.eval()
    loss_numerator = 0.0
    weight_sum = 0.0

    with torch.no_grad():
        for batch_idx in tensor_batch_indices(len(target), batch_size, X_macro.device, shuffle=False):
            b_macro = X_macro[batch_idx]
            b_micro = X_micro[batch_idx]
            b_industry = X_industry[batch_idx]
            b_y = target[batch_idx]
            b_w = weights[batch_idx]

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)

            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            batch_weight = float(b_w.sum().item())
            loss_numerator += loss.item() * batch_weight
            weight_sum += batch_weight

    return loss_numerator / max(weight_sum, 1e-12)

In [ ]:
# ==========================================
# 所有 MDN 共用的 Early-stopping 训练
# ==========================================
def train_mdn(
    model, X_train_macro, X_train_micro, X_train_industry, y_train, w_train, X_val_macro, 
    X_val_micro, X_val_industry, y_val, w_val, epochs=300, batch_size=1024, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
        mode="min", factor=0.5, patience=5, threshold=1e-4, threshold_mode="abs", min_lr=1e-6,
    )

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            len(y_train), batch_size, X_train_macro.device, shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro,b_industry,)

            loss = mdn_nll_loss_weighted(
                pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0,)
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        val_loss = evaluate_mdn(
            model, X_val_macro, X_val_micro, X_val_industry, y_val, w_val, batch_size,
        )
        if not np.isfinite(val_loss):
            raise RuntimeError("Validation NLL became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return model, best_val_loss, train_loss_history, val_loss_history

In [ ]:
# 使用 Train + Validation 确定 component 标签
def identify_components_from_trainval(model, datasets, inference_batch_size=8192):
    mu_parts, date_parts = [], []

    model.eval()
    with torch.no_grad():
        for X_macro, X_micro, X_industry, dates in datasets:
            for idx in tensor_batch_indices(len(dates), inference_batch_size, X_macro.device, shuffle=False):
                _, mu, _, _ = model(X_macro[idx], X_micro[idx], X_industry[idx])
                mu_parts.append(mu.cpu().numpy())

            date_parts.append(pd.to_datetime(dates).to_numpy())

    mu_all = np.concatenate(mu_parts)
    dates_all = np.concatenate(date_parts)

    # 先计算每月横截面平均，再对192个月等权平均
    mu_df = pd.DataFrame(mu_all)
    mu_df["mthcaldt"] = dates_all
    component_mean_mu = mu_df.groupby("mthcaldt").mean().mean().to_numpy()

    # 顺序：adverse、normal、favorable
    component_order = np.argsort(component_mean_mu)
    return component_mean_mu, component_order

In [ ]:
# ==========================================
# 所有 MDN 共用的滚动样本外预测
# ==========================================
def run_mdn_model(panel, model_class, model_name, num_components, 
                  dist_type="normal", model_kwargs=None,
):
    
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)

    model_kwargs = {} if model_kwargs is None else model_kwargs
    window_gen = make_window_generator(panel)
    global_X_mic, global_ind, global_y = make_device_tensors(panel)

    all_predictions = []
    history_logs = []
    embedding_history = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[{model_name} K={num_components} | Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        train_macro_z, val_macro_z, test_macro_z, macro_scaler = (
            standardize_macro_window(
                train_df,
                val_df,
                test_df,
                macro_cols,
            )
        )

        X_tr_mac = torch.from_numpy(
            train_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_va_mac = torch.from_numpy(
            val_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_te_mac = torch.from_numpy(
            test_macro_z.to_numpy(dtype=np.float32, copy=True)
        ).to(device)

        X_tr_mic, X_tr_ind, y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_y[idx_tr]
        X_va_mic, X_va_ind, y_va = global_X_mic[idx_va], global_ind[idx_va], global_y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]

        w_tr = torch.from_numpy(
            calculate_sample_weights(train_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(val_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        # 每个窗口独立初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = model_class(macro_dim=len(macro_cols), micro_dim=len(micro_cols), num_industries=50,
            num_components=num_components, dropout_rate=dropout_rate, dist_type=dist_type, **model_kwargs,
        ).to(device)

        model, best_val_loss, tr_hist, va_hist = train_mdn(
            model=model, X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, X_train_industry=X_tr_ind,
            y_train=y_tr, w_train=w_tr, X_val_macro=X_va_mac, X_val_micro=X_va_mic, X_val_industry=X_va_ind,
            y_val=y_va, w_val=w_va, epochs=early_stopping_max_epochs, batch_size=batch_size, lr=lr,
            patience=patience)

        model.eval()

        component_mean_mu = None
        component_order = None

        if model_name.startswith("structured") and num_components > 1:
            component_mean_mu, component_order = identify_components_from_trainval(
                model,
                datasets=[
                    (X_tr_mac, X_tr_mic, X_tr_ind, train_df["mthcaldt"]),
                    (X_va_mac, X_va_mic, X_va_ind, val_df["mthcaldt"]),
                ],
                inference_batch_size=8192,
            )

        with torch.no_grad():
            pi_logits, mu, sigma, nu = model(X_te_mac, X_te_mic, X_te_ind)
            pi = torch.softmax(pi_logits, dim=1)

        output = test_df[["permno", "mthcaldt", "target_ret_final"]].copy()

        output["window"] = window_idx

        if component_order is not None:
            output["adverse_component"] = int(component_order[0])
            output["normal_component"] = int(component_order[1])
            output["favorable_component"] = int(component_order[2])

        output["model_id"] = f"{model_name}_{dist_type}_k{num_components}"
        output["pi_vec"] = [json.dumps(row.tolist()) for row in pi.cpu().numpy()]
        output["pi_logits_vec"] = [json.dumps(row.tolist()) for row in pi_logits.cpu().numpy()]
        output["mu_vec"] = [json.dumps(row.tolist()) for row in mu.cpu().numpy()]
        output["sigma_vec"] = [json.dumps(row.tolist()) for row in sigma.cpu().numpy()]
        output["nu_vec"] = [json.dumps(row.tolist()) for row in nu.cpu().numpy()] if nu is not None else None

        all_predictions.append(output)

        embedding_weights = model.ind_embedding.weight.detach().cpu().numpy()
        window_embeddings = pd.DataFrame(
            embedding_weights, columns=[f"emb_dim_{i + 1}" for i in range(embedding_weights.shape[1])]
        )
        window_embeddings[industry_type] = window_embeddings.index
        window_embeddings["window"] = window_idx
        window_embeddings["test_period"] = f"{info['test'][0]} to {info['test'][1]}"
        embedding_history.append(window_embeddings)

        history_logs.append({
            "window": window_idx,
            "model_id": f"{model_name}_{dist_type}_k{num_components}",
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "best_epoch": int(np.argmin(va_hist) + 1),
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
            "component_mean_mu": (
                json.dumps(component_mean_mu.tolist())
                if component_mean_mu is not None else None
            ),
            "component_order": (
                json.dumps(component_order.tolist())
                if component_order is not None else None
            ),
            "adverse_component": (
                int(component_order[0])
                if component_order is not None else None
            ),
            "normal_component": (
                int(component_order[1])
                if component_order is not None else None
            ),
            "favorable_component": (
                int(component_order[2])
                if component_order is not None else None
            ),
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)
    embeddings = pd.concat(embedding_history, ignore_index=True)
    return forecasts, metrics, embeddings

In [ ]:
fused_k1_forecasts, fused_k1_metrics, fused_k1_embeddings = run_mdn_model(
    panel=processed_df,
    model_class=FusedMDN,
    model_name="fused",
    num_components=1,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)
# 执行保存
# fused_k1_forecasts.to_parquet(os.path.join(path, f'fused_k1_forecasts_{timestamp}.parquet'), index=False)
fused_k1_metrics.to_parquet(os.path.join(path, f'fused_k1_metrics_{timestamp}.parquet'), index=False)
fused_k1_embeddings.to_parquet(os.path.join(path, f'fused_k1_embeddings_{timestamp}.parquet'), index=False)


display(pd.concat([fused_k1_forecasts.head(4), fused_k1_forecasts.tail(4),]))
display(pd.concat([fused_k1_metrics.head(2), fused_k1_metrics.tail(2),]))
display(pd.concat([fused_k1_embeddings.head(4), fused_k1_embeddings.tail(4),]))

In [ ]:
fused_k3_forecasts, fused_k3_metrics, fused_k3_embeddings = run_mdn_model(
    processed_df,
    FusedMDN,
    "fused",
    num_components=3,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)

# 执行保存
# fused_k3_forecasts.to_parquet(os.path.join(path, f'fused_k3_forecasts_{timestamp}.parquet'), index=False)
fused_k3_metrics.to_parquet(os.path.join(path, f'fused_k3_metrics_{timestamp}.parquet'), index=False)
fused_k3_embeddings.to_parquet(os.path.join(path, f'fused_k3_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([fused_k3_forecasts.head(4), fused_k3_forecasts.tail(4),]))
display(pd.concat([fused_k3_metrics.head(2), fused_k3_metrics.tail(2),]))
display(pd.concat([fused_k3_embeddings.head(4), fused_k3_embeddings.tail(4),]))

### ==========================================
### Structure K
### ==========================================

In [ ]:
# ==========================================
# Structured MDN：Macro → pi，Micro + Industry → mu 和 sigma
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, macro_hidden_dim=32,
                 micro_hidden_dim=64, head_dim=8, num_components=3, dropout_rate=0.05,
                 dist_type="normal"):
        super().__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        self.ind_embedding = nn.Embedding(num_industries, ind_emb_dim)

        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim, macro_hidden_dim // 2),
            nn.LayerNorm(macro_hidden_dim // 2),
            nn.ELU(),
        )

        self.micro_net = nn.Sequential(
            nn.Linear(micro_dim + ind_emb_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(macro_hidden_dim // 2, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)
        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        h_macro = self.macro_net(x_macro)

        if self.pi_head is None:
            pi_logits = h_macro.new_zeros((len(h_macro), 1))
        else:
            pi_logits = self.pi_head(h_macro)

        industry_embedding = self.ind_embedding(x_industry)
        h_micro = self.micro_net(torch.cat([x_micro, industry_embedding], dim=1))

        mu = self.mu_head(h_micro)
        sigma = F.softplus(self.sigma_head(h_micro)) + sigma_floor
        nu = F.softplus(self.nu_head(h_micro)) + 2.01 if self.nu_head is not None else None

        return pi_logits, mu, sigma, nu

In [ ]:
# ==========================================
# 运行 Structured MDN
# ==========================================

structured_forecasts, structured_metrics, structured_embeddings = (
    run_mdn_model(
        processed_df, StructuredMDN, "structured", num_components=3, dist_type="normal", 
        model_kwargs={
        "ind_emb_dim": 6,
        "macro_hidden_dim": 16,
        "micro_hidden_dim": 64,
        "head_dim": 8,
        },
    )
)
structured_metrics.to_parquet(os.path.join(path, f'structured_metrics_{timestamp}.parquet'), index=False)
structured_embeddings.to_parquet(os.path.join(path, f'structured_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([structured_forecasts.head(4), structured_forecasts.tail(4)]))
display(pd.concat([structured_metrics.head(2), structured_metrics.tail(2)]))
display(pd.concat([structured_embeddings.head(4), structured_embeddings.tail(4)]))

一、FiLM 应该接在哪里
你目前的 Structured K3 是：
$$\pi_t=\operatorname{softmax}(g(z_t)),\qquad
h_{it}=f(x_{it},e_i)$$,

$$\mu_{it,k}=a_k(h_{it}),\qquad
\sigma_{it,k}=\operatorname{softplus}(b_k(h_{it}))+\sigma_{\min}.$$
其中 \(z_t\) 是宏观变量，\(x_{it}\) 是公司特征，\(e_i\) 是行业 embedding。
它允许宏观环境改变三个分量的权重，但在微观输入相同时，宏观环境不能直接改变各分量的均值和尺度。
FiLM 可以加在 micro_net 的输出和分布参数 heads 之间：

$$\widetilde h_{it}
=
[1+\Delta\gamma(z_t)]\odot h_{it}
+\beta(z_t).$$
再让均值、尺度 heads 使用 \(\widetilde h\)。这样，同一个公司特征在不同宏观环境下，可以对应不同的风险分布。FiLM 本身就是这种由条件输入生成逐特征缩放和平移的机制；金融场景下是否有效，需要你的样本外实验验证。


| 版本 | FiLM 影响的位置 | 用途 |
|---|---|---|
| `structured_film_sigma` | 只影响 σ | 优先测试宏观环境对波动和尾部尺度的作用 |
| `structured_film_all` | 同时影响 μ、σ | 测试更灵活的宏观—公司交互 |

In [ ]:
# ==========================================
# 根据 MDN 预测计算 VaR 和 ES
# ==========================================
def calculate_es_table(forecasts, alphas=(0.01, 0.05, 0.10),
                       chunk_size=50_000, max_rows=None, iterations=45):

    """
    根据一个模型的 MDN forecast DataFrame 计算左尾 VaR 和收益率 ES。

    var_alpha = F^{-1}(alpha)，是收益率分位数；
    es_alpha  = E[R | R <= var_alpha]，亏损保留负数，不取绝对值。

    每次只能传入一个 model_id，但可以是 K=1、K=3、Normal 或 Student-t。
    """
    frame = forecasts.iloc[:max_rows] if max_rows is not None else forecasts
    alphas = np.asarray(alphas, dtype=float)

    def decode(column):
        values = frame[column].map(lambda x: json.loads(x) if isinstance(x, str) else x)
        return np.stack(values).astype(float, copy=False)

    pi = decode("pi_vec")
    mu = decode("mu_vec")
    sigma = decode("sigma_vec")

    if pi.shape != mu.shape or mu.shape != sigma.shape:
        raise ValueError("pi、mu 和 sigma 的维度不一致。")
    if not np.isfinite(pi).all() or not np.isfinite(mu).all() or not np.isfinite(sigma).all():
        raise ValueError("pi、mu 或 sigma 含有 NaN/inf。")
    if np.any(sigma <= 0):
        raise ValueError("sigma 必须大于0。")
    if not np.allclose(pi.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError("部分 mixture probabilities 加总不等于1。")

    # Normal 模型的 nu_vec 全为空；Student-t 模型的 nu_vec 应全部非空
    nu = None
    if "nu_vec" in frame.columns:
        has_nu = frame["nu_vec"].notna()

        if has_nu.any() and not has_nu.all():
            raise ValueError("nu_vec 不能同时包含空值和非空值。")

        if has_nu.all():
            nu = decode("nu_vec")
            if nu.shape != mu.shape or not np.isfinite(nu).all() or np.any(nu <= 2):
                raise ValueError("nu 必须与 mu 维度一致，而且全部大于2。")
            
    n_rows, n_components = pi.shape
    n_alphas = len(alphas)

    var_matrix = np.empty((n_rows, n_alphas), dtype=float)
    es_matrix = np.empty((n_rows, n_alphas), dtype=float)
    bracket_probability = max(float(alphas.min()) * 0.1, 1e-10)

    for start in range(0, n_rows, chunk_size):
        stop = min(start + chunk_size, n_rows)

        p = pi[start:stop]
        m = mu[start:stop]
        s = sigma[start:stop]
        degrees = None if nu is None else nu[start:stop]

        # Normal K=1 有闭式解，不需要二分法
        if degrees is None and n_components == 1:
            z = norm.ppf(alphas)[None, :]
            chunk_var = m + s * z
            chunk_es = m - s * norm.pdf(z) / alphas[None, :]

        else:
            # 为每一行建立足够宽的分位数搜索区间
            if degrees is None:
                lower_component = m + s * norm.ppf(bracket_probability)
                upper_component = m + s * norm.ppf(1.0 - bracket_probability)
            else:
                lower_component = m + s * student_t.ppf(bracket_probability, df=degrees)
                upper_component = m + s * student_t.ppf(1.0 - bracket_probability, df=degrees)

            lower = np.repeat(lower_component.min(axis=1)[:, None], n_alphas, axis=1)
            upper = np.repeat(upper_component.max(axis=1)[:, None], n_alphas, axis=1)

            # 批量二分法求 mixture quantiles
            for _ in range(iterations):
                middle = (lower + upper) / 2.0
                z = (middle[..., None] - m[:, None, :]) / s[:, None, :]

                if degrees is None:
                    component_cdf = norm.cdf(z)
                else:
                    component_cdf = student_t.cdf(z, df=degrees[:, None, :])

                mixture_cdf = np.sum(p[:, None, :] * component_cdf, axis=2)
                lower = np.where(mixture_cdf < alphas, middle, lower)
                upper = np.where(mixture_cdf >= alphas, middle, upper)

            chunk_var = (lower + upper) / 2.0
            z = (chunk_var[..., None] - m[:, None, :]) / s[:, None, :]

            if degrees is None:
                component_cdf = norm.cdf(z)
                adjusted_pdf = norm.pdf(z)
            else:
                degrees_3d = degrees[:, None, :]
                component_cdf = student_t.cdf(z, df=degrees_3d)
                adjusted_pdf = (
                    (degrees_3d + z**2) / (degrees_3d - 1.0)
                    * student_t.pdf(z, df=degrees_3d)
                )

            left_moment = (
                m[:, None, :] * component_cdf
                - s[:, None, :] * adjusted_pdf
            )

            chunk_es = (
                np.sum(p[:, None, :] * left_moment, axis=2)
                / alphas[None, :]
            )

        var_matrix[start:stop] = chunk_var
        es_matrix[start:stop] = chunk_es

    # Mixture conditional mean and standard deviation
    mean_pred = np.sum(pi * mu, axis=1)

    if nu is None:
        component_variance = sigma**2
    else:
        component_variance = sigma**2 * nu / (nu - 2.0)

    mixture_variance = (
        np.sum(pi * (mu**2 + component_variance), axis=1)
        - mean_pred**2
    )
    vol_pred = np.sqrt(np.maximum(mixture_variance, 0.0))

    result = frame[["permno", "mthcaldt", "model_id"]].reset_index(drop=True)

    for column, alpha in enumerate(alphas):
        suffix = f"{alpha * 100:g}".replace(".", "_")
        result[f"es_{suffix}"] = es_matrix[:, column]
        result[f"var_{suffix}"] = var_matrix[:, column]

    result["mean_pred"] = mean_pred
    result["vol_pred"] = vol_pred
    result["realized_ret"] = frame["target_ret_final"].to_numpy(dtype=float)
    
    return result

In [ ]:
# ==========================================
# 计算 VaR/ES，合并回原始 forecasts，再保存
# ==========================================
def attach_var_es(forecasts):
    risk = calculate_es_table(forecasts, alphas=ALPHAS, max_rows=None)

    keys = ["permno", "mthcaldt", "model_id"]
    risk_cols = [
        f"{name}_{ALPHA_SUFFIX[alpha]}"
        for alpha in ALPHAS
        for name in ("var", "es")
    ]

    # 防止重复运行时产生 var_1_x、var_1_y
    forecasts = forecasts.drop(columns=risk_cols, errors="ignore")

    combined = forecasts.merge(
        risk[keys + risk_cols],
        on=keys,
        how="left",
        validate="one_to_one",
    )

    # if combined[risk_cols].isna().any().any():
    #     raise ValueError("部分 forecasts 没有匹配到 VaR/ES。")

    return combined


max_rows = None

fused_k1_forecasts = attach_var_es(fused_k1_forecasts)
fused_k3_forecasts = attach_var_es(fused_k3_forecasts)
structured_forecasts = attach_var_es(structured_forecasts)
# film_sigma_forecasts = attach_var_es(film_sigma_forecasts)
# film_all_forecasts = attach_var_es(film_all_forecasts)


forecast_files = {
    "fused_k1": fused_k1_forecasts,
    "fused_k3": fused_k3_forecasts,
    "structured": structured_forecasts,
    # "film_sigma": film_sigma_forecasts,
    # "film_all": film_all_forecasts,
}

for name, forecasts in forecast_files.items():
    forecasts.to_parquet(
        os.path.join(path, f"{name}_forecasts_{timestamp}.parquet"),
        index=False,
    )

display(fused_k1_forecasts.head(2))
display(fused_k3_forecasts.head(2))
display(structured_forecasts.head(2))
# display(film_sigma_forecasts.head(2))
# display(film_all_forecasts.head(2))

In [ ]:
# # run_test = "full"  # "quick" 或 "full"
# max_rows = 100 if run_test == "quick" else None
# alphas = (0.01, 0.05, 0.10)

# fused_k1_es = calculate_es_table(
#     fused_k1_forecasts, alphas=alphas, max_rows=max_rows
# )
# fused_k1_es.to_parquet(
#     os.path.join(path, f"ES_fused_normal_k1_{timestamp}.parquet"), index=False
# )

# fused_k3_es = calculate_es_table(
#     fused_k3_forecasts, alphas=alphas, max_rows=max_rows
# )
# fused_k3_es.to_parquet(
#     os.path.join(path, f"ES_fused_normal_k3_{timestamp}.parquet"), index=False
# )

# structured_es = calculate_es_table(
#     structured_forecasts, alphas=alphas, max_rows=max_rows
# )
# structured_es.to_parquet(
#     os.path.join(path, f"ES_structured_normal_k3_{timestamp}.parquet"), index=False
# )

# display(pd.concat([fused_k1_es.head(2), fused_k3_es.tail(2)], ignore_index=True))
# display(pd.concat([structured_es.head(2), structured_es.tail(2)], ignore_index=True))
# display(pd.concat([fused_k1_es.head(2), structured_es.head(2)], ignore_index=True))

In [ ]:
def summarize_training(metrics):
    rows = []

    for _, row in metrics.iterrows():
        train_loss = np.asarray(row["train_loss_trace"], dtype=float)
        val_loss = np.asarray(row["val_loss_trace"], dtype=float)
        best_idx = np.argmin(val_loss)

        rows.append({
            "window": row["window"],
            "model_id": row["model_id"],
            "epochs_run": len(val_loss),
            "best_epoch": best_idx + 1,
            "train_first": train_loss[0],
            "train_best_epoch": train_loss[best_idx],
            "train_last": train_loss[-1],
            "val_first": val_loss[0],
            "val_best": val_loss[best_idx],
            "val_last": val_loss[-1],
        })

    return pd.DataFrame(rows)


display(summarize_training(fused_k1_metrics))
display(summarize_training(fused_k3_metrics))
display(summarize_training(structured_metrics))
# display(summarize_training(film_sigma_metrics))
# display(summarize_training(film_all_metrics))

In [ ]:
def mixture_diagnostics(forecasts):
    pi = np.stack(
        forecasts["pi_vec"].map(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
    )

    return pd.Series({
        "mean_effective_k": np.mean(1.0 / np.sum(pi**2, axis=1)),
        "mean_max_pi": np.mean(pi.max(axis=1)),
        "share_max_pi_above_0_95": np.mean(pi.max(axis=1) > 0.95),
    })


display(mixture_diagnostics(fused_k3_forecasts))
display(mixture_diagnostics(structured_forecasts))
# display(mixture_diagnostics(film_sigma_forecasts))
# display(mixture_diagnostics(film_all_forecasts))

In [ ]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()